# Introduction to backpropagation and gradient descent

We start with a very, very simple machine learning problem: converting measuremens in feet to meters. The mathematical model is simple, and contains a single parameter $w$:

$$ y = wx $$

where $x$ is the length in feet, and $y$ is the length in meters. In this case, we can look up what the value of $w$ should be: 0.3048. This is practical to check out results, but for now let us pretend that we do not know the value of $w$.

Instead, we have access to a table of measurements that are done in both feet and meters. We load this using NumPy's `loadtxt` function:

In [1]:
import numpy as np
data = np.loadtxt("feet_meters.tsv")

data[:10]

array([[77.5, 23.6],
       [54.2, 16.5],
       [89.1, 27.2],
       [54.4, 16.6],
       [73.8, 22.5],
       [22.2,  6.8],
       [26. ,  7.9],
       [ 6.6,  2. ],
       [76.7, 23.4],
       [18. ,  5.5]])

Note that the values are rounded, which means that we do not get exactly the correct result if we multiply the measurement in feet by 0.3048:

In [2]:
print(f"{data[0,0]}*0.3048 = {data[0,0]*0.3048} (measurement is {data[0,1]})")

77.5*0.3048 = 23.622 (measurement is 23.6)


If we know $w$, it is easy to write a function that predicts the length in meters given the length in feet:

In [3]:
def predict(x, w):
    return w*x

Our goal is to find a value for $w$ that makes the `predict` function work as well as possible on the data we have. To quantify what we mean by "working well", we need to provide a *loss function* which tells us how wrong the predictions get for a given value of $w$.

There are many ways of choosing a loss function. One way that is both mathematically convenient and works well in practice, is to less the loss be defined as the squared difference between our prediction and the actual value. We code this as follows:

In [4]:
def loss(x, y, w):
    return (predict(x, w) - y) ** 2

For instance, even with $w = 0.3048$ the first datapoints gives us some loss, since the answer in our data (which we by definition assume is correct) has some rounding error in it. This may sound like a serious problem, but it turns out that having small random errors in your data is actually beneficial during learning. If we use a value of $w$ this is very wrong, say $w = 1$, the loss is considerably higher.

In [5]:
print(loss(data[0,0], data[0,1], 0.3048))
print(loss(data[0,0], data[0,1], 1.0))

0.00048399999999993246
2905.21


If we look at another point, the corresponding losses will be different because the measurements are at a different scale, and the rounding error is also somewhat different.

In [6]:
print(loss(data[1,0], data[1,1], 0.3048))
print(loss(data[1,0], data[1,1], 1.0))

0.00040642560000002506
1421.2900000000002


*Stochastic gradient descent* works by looking at one (or a few) example at a time, and considering what happens if we wiggle the parameter(s) a little, then adjusting them to give a lower loss. In our case, we only have one parameter, $w$.

The whole algorithm is very simple. We just need to provide a starting value for $w$ (here we use a value of 1), then repeatedly try data points, and adjust $w$ to make the loss a little bit lower in each instance.

We try this manually for a few steps first.

In [7]:
# Initial value of the parameter w
w = 1.0
# How much to "wiggle" the parameter
dw = 0.01

print(loss(data[1,0], data[1,1], w - dw))
print(loss(data[1,0], data[1,1], w + dw))

1380.7169640000002
1462.4505640000004


When increasing $w$ by a small amount, our loss increased. Since we want to decrease the loss, we should not increase but rather *decrease* $w$. Let us try to decrease it to 0.5.

In [8]:
w = 0.5
print(loss(data[1,0], data[1,1], w - dw))
print(loss(data[1,0], data[1,1], w + dw))

101.163364
124.14416400000006


The loss is considerably better (lower) now, but increasing it by a small amount still makes it worse, so we need to decrease further. Since the difference when "wiggling" $w$ is smaller now than last time, we probably do not need to decrease it as much as 0.5 this time. Let us try to decrease by 0.25 instead.

In [9]:
w = 0.25
print(loss(data[1,0], data[1,1], w - dw))
print(loss(data[1,0], data[1,1], w + dw))

12.194063999999994
5.798463999999997


Even better! But this time the loss goes down when increasing $w$ a little, which means that now we should *increase* rather than decrease $w$. Also, the difference is quite low, so let us try a small step and increase to 0.3.

In [10]:
w = 0.3
print(loss(data[1,0], data[1,1], w - dw))
print(loss(data[1,0], data[1,1], w + dw))

0.6115240000000001
0.09120399999999976


Now we are getting really close. Recall that the "correct" value is 0.3048.

Typically one would proceed like this until some stopping criterion has been met, for instance having seen all the data points, or having a loss that is low enough (when averaged over several data points).

## Step 1: machine learning with middle age math

The above method is really how training of neural networks is fundamentally done, but some additional tricks are needed in order to be practical.

To begin with, we have so far been using intuition when deciding how *much* to change the parameter $w$ at each step. In practice, one would let the size of the update be proportional to how much the loss changes when we wiggle $w$ by a small amount. Since the change depends on exactly how small this amount ($\Delta w$, the Python variable `dw`) is, we normalize by dividing by $2\Delta w$. Let us make this little modification, to automate the process.

In [11]:
def print_info(x, y, w):    
    print(f"w = {w:.4g}, loss(w+{dw}) = {loss(x, y, w + dw):.4g}, loss(w-{dw}) = {loss(x, y, w - dw):.4g}")

w = 1.0
for x, y in data[:10]:
    print_info(x, y, w)
    dloss_dw = (loss(x, y, w + dw) - loss(x, y, w - dw)) / (2*dw)
    w = w - dloss_dw

w = 1, loss(w+0.01) = 2989, loss(w-0.01) = 2822
w = -8354, loss(w+0.01) = 2.05e+11, loss(w-0.01) = 2.05e+11
w = 4.907e+07, loss(w+0.01) = 1.912e+19, loss(w-0.01) = 1.912e+19
w = -7.791e+11, loss(w+0.01) = 1.796e+27, loss(w-0.01) = 1.796e+27
w = 4.603e+15, loss(w+0.01) = 1.154e+35, loss(w-0.01) = 1.154e+35
w = 4.603e+15, loss(w+0.01) = 1.044e+34, loss(w-0.01) = 1.044e+34
w = 4.603e+15, loss(w+0.01) = 1.433e+34, loss(w-0.01) = 1.433e+34
w = 4.603e+15, loss(w+0.01) = 9.231e+32, loss(w-0.01) = 9.231e+32
w = 4.603e+15, loss(w+0.01) = 1.247e+35, loss(w-0.01) = 1.247e+35
w = 4.603e+15, loss(w+0.01) = 6.866e+33, loss(w-0.01) = 6.866e+33


Oops! What went wrong here?

The problem is that the steps we took were way too large. This can easily be fixed, by scaling down the step size by a constant factor. For instance, we can take only 1% of the value in `dloss_dw`. This is what is usually referred to the *learning rate*.

In [12]:
w = 1.0
learning_rate = 0.01
for x, y in data[:10]:
    print_info(x, y, w)
    dloss_dw = (loss(x, y, w + dw) - loss(x, y, w - dw)) / (2*dw)
    w = w - learning_rate*dloss_dw

w = 1, loss(w+0.01) = 2989, loss(w-0.01) = 2822
w = -82.55, loss(w+0.01) = 2.016e+07, loss(w-0.01) = 2.017e+07
w = 4785, loss(w+0.01) = 1.818e+11, loss(w-0.01) = 1.818e+11
w = -7.549e+05, loss(w+0.01) = 1.687e+15, loss(w-0.01) = 1.687e+15
w = 4.393e+07, loss(w+0.01) = 1.051e+19, loss(w-0.01) = 1.051e+19
w = -4.741e+09, loss(w+0.01) = 1.108e+22, loss(w-0.01) = 1.108e+22
w = 4.199e+10, loss(w+0.01) = 1.192e+24, loss(w-0.01) = 1.192e+24
w = -5.257e+11, loss(w+0.01) = 1.204e+25, loss(w-0.01) = 1.204e+25
w = -6.827e+10, loss(w+0.01) = 2.742e+25, loss(w-0.01) = 2.742e+25
w = 7.965e+12, loss(w+0.01) = 2.056e+28, loss(w-0.01) = 2.056e+28


Still too large, it seems. Let us try to scale down the learning rate further.

In [13]:
w = 1.0
learning_rate = 0.0001
for x, y in data[:10]:
    print_info(x, y, w)
    dloss_dw = (loss(x, y, w + dw) - loss(x, y, w - dw)) / (2*dw)
    w = w - learning_rate*dloss_dw

w = 1, loss(w+0.01) = 2989, loss(w-0.01) = 2822
w = 0.1645, loss(w+0.01) = 49.55, loss(w-0.01) = 65.99
w = 0.2467, loss(w+0.01) = 18.71, loss(w-0.01) = 37.3
w = 0.3397, loss(w+0.01) = 5.87, loss(w-0.01) = 1.782
w = 0.3192, loss(w+0.01) = 3.233, loss(w-0.01) = 0.1038
w = 0.3036, loss(w+0.01) = 0.02619, loss(w-0.01) = 0.07963
w = 0.3039, loss(w+0.01) = 0.06782, loss(w-0.01) = 0.06738
w = 0.3039, loss(w+0.01) = 0.005109, loss(w-0.01) = 0.003663
w = 0.3039, loss(w+0.01) = 0.4523, loss(w-0.01) = 0.7421
w = 0.3053, loss(w+0.01) = 0.03078, loss(w-0.01) = 0.03406


Now things seem at least don't explode during the first ten datapoints, so we can try with the rest of the data.

In [14]:
for x, y in data[10:-10]:
    dloss_dw = (loss(x, y, w + dw) - loss(x, y, w - dw)) / (2*dw)
    w = w - learning_rate*dloss_dw

for x, y in data[-10:]:
    print_info(x, y, w)
    dloss_dw = (loss(x, y, w + dw) - loss(x, y, w - dw)) / (2*dw)
    w = w - learning_rate*dloss_dw

w = 0.3048, loss(w+0.01) = 0.2862, loss(w-0.01) = 0.2927
w = 0.3048, loss(w+0.01) = 0.1316, loss(w-0.01) = 0.1578
w = 0.3049, loss(w+0.01) = 0.0002969, loss(w-0.01) = 0.005295
w = 0.305, loss(w+0.01) = 0.5083, loss(w-0.01) = 0.5792
w = 0.3053, loss(w+0.01) = 0.05704, loss(w-0.01) = 0.01567
w = 0.3051, loss(w+0.01) = 0.003812, loss(w-0.01) = 0.0003896
w = 0.3051, loss(w+0.01) = 0.3958, loss(w-0.01) = 0.4288
w = 0.3053, loss(w+0.01) = 0.7417, loss(w-0.01) = 0.6477
w = 0.3048, loss(w+0.01) = 0.06887, loss(w-0.01) = 0.03013
w = 0.3046, loss(w+0.01) = 0.2964, loss(w-0.01) = 0.2262


Success! Due to the rounding errors, the value of $w$ will vary a little bit up and down, but not very far from 0.3048. Note however that the loss still varies widely. This is quite typical, when looking at individual datapoints. Often one computes the average loss over a *batch* of a small number of datapoints, in order to reduce these fluctuations and make learning more efficient.

# Step 2: derivatives

While the above approach works, the method of wiggling $w$ back and forth in order to find out how much to increase or decrease it, is cumbersome and slow.

The key trick, developed by Isaac Newton and Gottfried Wilhelm Leibniz over 300 years ago, is to consider what happens when $\Delta w$ decreases to a smaller and smaller value. It turns out that as long as we know the equation for computing the loss, we can directly derive a corresponding equation for how much the loss varies as we wiggle $w$. In our case, the loss is computed as
$$ L = (wx - y)^2 $$
When wiggling $w$ by a really tiny amount, it turns out that the (normalized) change of the loss $L$ is
$$ \frac{\mathrm{d}L}{\mathrm{d}w} = 2x(wx-y) $$
In case you have studied calculus, you may remember that there are two ways of finding this. Either we expand the square to make it a simple polynomial of $w$:
$$ L = (wx - y)^2 = w^2x^2 - 2wxy + y^2 $$
which gives
$$ \frac{\mathrm{d}L}{\mathrm{d}w} = 2wx^2 - 2xy = 2x(wx - y) $$
Alternatively, and this is key to moving forward later, we can use the *chain rule* which states that
$$ \frac{\mathrm{d}c}{\mathrm{d}a} = \frac{\mathrm{d}c}{\mathrm{d}b} \cdot \frac{\mathrm{d}b}{\mathrm{d}a} $$
If we let $c = b^2 = L$ and $b = wx-y$ and $a = w$, we get
$$ \frac{\mathrm{d}L}{\mathrm{d}w} = \frac{\mathrm{d}b^2}{\mathrm{d}b} \cdot \frac{\mathrm{d}(wx-y)}{\mathrm{d}w} = 2b \cdot x = 2x(wx-y) $$

By applying the chain rule repeatedly, we can easily compute the derivative of the loss with respect to a parameter even in very complex expressions, such as the equation for an LLM.

Now, we can try to compute `dloss_dt` in this simpler fashion. 

In [15]:
w = 1.0
for x, y in data[:-10]:
    dloss_dw = 2*x*(w*x-y)
    w = w - learning_rate*dloss_dw

for x, y in data[-10:]:
    print(f"w = {w:.4f}")
    dloss_dw = 2*x*(w*x-y)
    w = w - learning_rate*dloss_dw

w = 0.3048
w = 0.3048
w = 0.3049
w = 0.3050
w = 0.3053
w = 0.3051
w = 0.3051
w = 0.3053
w = 0.3048
w = 0.3046


It works! And the results are identical to what we arrived at earlier. In general this way is in fact more accurate since the old way of computing `dloss_dw` is just an approximation, but in this case we have a very simple linear function so the approximation happens to be exact.

## Step 3: vectors, feet and inches

So far we have worked with only a single parameter: the ratio between feet and meters. Next step is to try a somewhat more complex model with *two* parameters, whose task is to convert measurements in feet and inches to meters. We assume that the number of meters can be expressed as
$$ x_0w_0 + x_1w_1 $$
We now have two-dimensional inputs, containing $x_0$ (the number of feet) and $x_1$ (the number of inches), but still only one $y$ representing the number of meters. We use one parameter $w_0$ (meters per foot) and one parameter $w_1$ (meters per inch).

As usual, start with having a peek at the data.

In [16]:
data2 = np.loadtxt("feet_inches_meters.tsv")

data2[:10]

array([[63.  ,  7.  , 19.38],
       [79.  , 10.  , 24.33],
       [65.  ,  2.  , 19.86],
       [21.  ,  5.  ,  6.53],
       [18.  , 10.  ,  5.74],
       [75.  ,  9.  , 23.09],
       [95.  , 10.  , 29.21],
       [16.  ,  4.  ,  4.98],
       [46.  , 11.  , 14.3 ],
       [64.  ,  6.  , 19.66]])

The columns represent feet, inches and meters, in that order.

Now, since our equation for $y$ is different from before, we also need to update our definition of the loss.

$$ L = (x_0w_0 + x_1w_1 - y)^2 $$

Now, you can use either expand the square (if you like long expressions) or use the chain rule (somewhat easier) to find the derivatives of the loss with respect to $w_0$ and $w_1$.
$$ \begin{aligned}
\frac{\mathrm{d}L}{\mathrm{d}w_0} &= 2x_0(w_0x_0 + w_1x_1 - y) \\
\frac{\mathrm{d}L}{\mathrm{d}w_1} &= 2x_1(w_0x_0 + w_1x_1 - y)
\end{aligned} $$
If we put these together in a vector, we get the *gradient* for $L$ with respect to the parameter *vector* $\mathbf{w}$.
$$ \nabla L = 2\mathbf{x}(\mathbf{w}\cdot\mathbf{x} - y) $$
Apart from $\mathbf{w}$ and $\mathbf{x}$ being vectors instead of scalars, this is exactly what we arrived at in Step 2. In other words, when using NumPy we hardly have to modify the code when we change from one to two parameters in our model:

In [17]:
w = np.array([1.0, 1.0])

def predict(x, w):
    # Instead of multiplying, we use the dot product here
    return w @ x

for i, (x, y) in enumerate(zip(data2[:,:2], data2[:,2])):
    dloss_dw = 2*x*(predict(x, w) - y)
    w = w - learning_rate*dloss_dw
    if i % 100 == 0:
        print(f"w = {np.round(w, 4)}")

w = [0.3622 0.9291]
w = [0.2488 0.6192]
w = [0.2905 0.4242]
w = [0.2893 0.268 ]
w = [0.2977 0.1779]
w = [0.2929 0.1277]
w = [0.2992 0.0966]
w = [0.2977 0.0732]
w = [0.3038 0.0574]
w = [0.3016 0.0454]


Looking OK(ish) for the feet ($w_0$) but not quite there yet for the inches ($w_1$), which should be $0.0254$. We try training for a while longer on the same data.

In [18]:
for epoch in range(1, 4):
    for x, y in zip(data2[:,:2], data2[:,2]):
        dloss_dw = 2*x*(predict(x, w) - y)
        w = w - learning_rate*dloss_dw
    print(f"w = {np.round(w, 4)} after epoch {epoch}")

w = [0.3049 0.0257] after epoch 1
w = [0.3049 0.0254] after epoch 2
w = [0.3049 0.0254] after epoch 3


So, it turns out we needed another pass through the data in order to properly train the model.